This notebook is supplementary material of LLMs on a Budget's first chapter.

It shows how to fine-tune an LLM with LoRA, using Unsloth.

*Last Update: September 17th, 2025*

#How to use this notebook

Run all the cell.

#Hardware Requirements
This notebook requires a GPU with at least 6 GB of VRAM. It has been tested with a GPU supporting bfloat16 and FlashAttention. I recommend an Ampere GPU, or more recent. Some examples of compatible GPUs:

* All the RTX GPUs
* All the AXX, e.g., A40, A100, GPUs
* H100


#Software Requirements

This notebook has been tested with CUDA 12.6 and Pytorch 2.8.0. It should also work with more recent versions. If you have compatibility issues, let me know.


Unsloth is optimized for specific versions of CUDA and Pytorch. To find which version of Unsloth you should use, run this command:

In [ ]:
!wget -qO- https://raw.githubusercontent.com/unslothai/unsloth/main/unsloth/_auto_install.py | python -

pip install --upgrade pip && pip install --no-deps git+https://github.com/unslothai/unsloth-zoo.git && pip install "unsloth[cu128-torch2100] @ git+https://github.com/unslothai/unsloth.git" --no-build-isolation


Then, we can install it:

In [ ]:
!pip install --upgrade pip && pip install --no-deps git+https://github.com/unslothai/unsloth-zoo.git && pip install "unsloth[cu128-torch2100] @ git+https://github.com/unslothai/unsloth.git" --no-build-isolation

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-lztuzpv8/unsloth_71d380a70165401b935c1f9d2a65cb15
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-lztuzpv8/unsloth_71d380a70165401b935c1f9d2a65cb15
  Resolved https://github.com/unslothai/unsloth.git to commit 41a6cc8692bd47ba0e4590b45bdf88d24b15ef33
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> unsloth

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:
!pip install --upgrade pip &&

!pip install --no-deps git+https://github.com/unslothai/unsloth-zoo.git &&

!pip install "unsloth[cu128-torch2100] @ git+https://github.com/unslothai/unsloth.git"

/bin/bash: -c: line 2: syntax error: unexpected end of file
/bin/bash: -c: line 2: syntax error: unexpected end of file
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-0fcw5ksw/unsloth_c05b136eb72c4baa94fa62aa5681a5e6
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-0fcw5ksw/unsloth_c05b136eb72c4baa94fa62aa5681a5e6
  Resolved https://github.com/unslothai/unsloth.git to commit 41a6cc8692bd47ba0e4590b45bdf88d24b15ef33
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 MB 67.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.7 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 139.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

#Fine-tuning Step-by-step

###Import packages
* **FastLanguageModel**: Load the tokenizer and model, and make the PEFT model
* **torch**: We will only use this package for data types such as torch.bfloat16
* **multiprocessing**: This package is used for parallel processing of the dataset. It is used to efficiently add the EOS token to all the training examples.

For this notebook, the fine-tuning data are stored on the Hugging Face. We directly load it with **load_dataset** from the datasets package.

To simplify fine-tuning, we use TRL's **SFTTrainer**. It automatically processes the data without the need to create a data collator. **SFTConfig** manages all the hyperparameters. It works similarly to TrainingArguments but contains a few more hyperparameters specific to supervised fine-tuning.

In [ ]:
from unsloth import FastLanguageModel
import torch, multiprocessing
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Load the model and tokenizer

FastLanguageModel loads both the tokenizer and model at the same time. It also optimizes the model to make it fast and memory efficient. Leave dtype to None as it can be automatically set by Unsloth given your GPU.

Increase max_seq_length if you want your model to be able to accurately process longer sequences. Note that it will increase memory consumption.

In [ ]:
max_seq_length = 512
model_name = "Qwen/Qwen2.5-1.5B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = False,
)

==((====))==  Unsloth 2026.4.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/Qwen2.5-1.5B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


##Load and Process the Training Data
For this notebook, I chose timdettmers/openassistant-guanaco. It's also a Hugging Face repository.

This is a multilingual instruction dataset. Each example contains an instruction paired with a correct answer. We can use this dataset to fine-tune Qwen2.5 to answer instructions.

This dataset is rather small. Don't expect the fine-tuned model to be very good. However, because it is small, it's a good dataset for testing/debugging a training pipeline.

In [ ]:
ds = load_dataset("timdettmers/openassistant-guanaco")
def process(row):
    row["text"] = row["text"]+tokenizer.eos_token
    return row

ds = ds.map(
    process,
    num_proc= multiprocessing.cpu_count(),
    load_from_cache_file=False,
)

README.md:   0%|          | 0.00/395 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


openassistant_best_replies_train.jsonl:   0%|          | 0.00/20.9M [00:00<?, ?B/s]

openassistant_best_replies_eval.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/9846 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/518 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/9846 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/518 [00:00<?, ? examples/s]

Next, we set LoRA's hyperparameters. For this, we use FastLanguageModel.get_peft_model which is similar to PEFT's LoraConfig but applies optimizations.

This configuration set the rank=alpha to get a LoRA scaling factor of 1.

* lora_alpha: The value for the alpha of LoRA
* r: The rank of LoRA's tensors. A higher rank consumes more memory but might yield better results
* lora_dropout: The dropout rate. Reduce it if LoRA doesn't seem to learn much from the data (e.g., if the training loss remains stable), or increase it if LoRA quickly overfits (e.g., if the training loss quickly decreases while the validation loss remains stable or increases).
* bias: For LoRA applied to LLMs, we rarely need to use a bias.
* use_gradient_checkpointing: Set to True to significantly reduce the memory consumption of the activations. You can also set to "unsloth" to activate Unsloth's optimizations for long context.
* target_modules: The list of the modules of the model that will be fine-tune with a LoRA adapter. Adding more modules usually lead to better results but also consumes more memory. In this examples, all the MLP and self-attention modules are targeted.
* random_state: Not mandatory but ensures reproducibility.

In [ ]:
# Do model patching and add fast LoRA weights
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 3407,
    max_seq_length = max_seq_length
)


Unsloth 2026.4.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


##Training Arguments

We configure all the remaining training hyperparameters with SFTConfig. I directly commented the code.

In [ ]:
training_arguments = SFTConfig(
        #The fine-tuned adapter will be saved in this directory
        output_dir="./Qwen2.5_1.5B_LoRA",

        #For optimization, we use AdamW with the optimizer states quantized to 8-bit
        optim="adamw_8bit",

        #To save as much memory as possible, we use batch size of 1. Since we use gradient_accumulation_steps of 16, the real training batch size is 16 (16*1). Increase per_device_train_batch_size if you have more memory, and decrease proportionnally gradient_accumulation_steps (unless the performance is not as good).
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        per_device_eval_batch_size=1,

        #The learning rate is 1e-4, It is always good to try different values. For LLMs, usually target values between 5e-6 and 3e-4.
        learning_rate=1e-4,

        #We train for 1 epoch. All the training example will be seen once.
        num_train_epochs=1,

        #The learning rate will reach its maximum value when 10% will have been completed, and then will linearly decrease
        warmup_ratio=0.1,
        lr_scheduler_type="linear",

        #In the fine-tuning dataset, we only use the "text" column
        dataset_text_field="text",

        #The maximum length of the sequence is 512 tokens. Examples longer that 512 will be truncated. Examples shorter than 512 will be padded.
        max_seq_length=512,

        #We train with bfloat16 parameters. The adapter will serialized with bfloat16. Remove this line if your GPU doesn't support bfloat16
        bf16 = False, # Changed from True to False as the GPU does not support bf16

        #The adapter will be saved after the first (and only) epoch is completed
        save_strategy="epoch",

        #Verbous logs. The training loss and other information will be printed every 25 steps.
        log_level="debug",
        logging_steps=25,
        #The validation split will be process every 25 steps to compute the validation loss
        eval_steps=25,
        eval_strategy="steps",
        do_eval=True,
        report_to="none"

)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
PyTorch: setting up devices


The SFTTrainer packs everything together: model, datasets, tokenizer, LoRA configuration, and the training arguments.

In [ ]:
trainer = SFTTrainer(
        model=model,
        train_dataset=ds['train'],
        eval_dataset=ds['test'],
        tokenizer=tokenizer,
        args=training_arguments,
)

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


##Training

Finally, we can start training!

In [ ]:
trainer.train()

skipped Embedding(151936, 1536, padding_idx=151665): 222.5625M params
bitsandbytes: will optimize Embedding(151936, 1536, padding_idx=151665) in fp32
skipped: 222.5625M params
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,846 | Num Epochs = 1 | Total steps = 616
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss,Validation Loss
25,1.534206,1.630321
50,1.472447,1.627065
75,1.426582,1.626150
100,1.425834,1.622167
125,1.374911,1.618845
150,1.477939,1.617234
175,1.471930,1.615543
200,1.480667,1.613480
225,1.477054,1.614026
250,1.480441,1.611485



***** Running Evaluation *****
  Num examples = 518
  Batch size = 1

***** Running Evaluation *****
  Num examples = 518
  Batch size = 1

***** Running Evaluation *****
  Num examples = 518
  Batch size = 1

***** Running Evaluation *****
  Num examples = 518
  Batch size = 1

***** Running Evaluation *****
  Num examples = 518
  Batch size = 1

***** Running Evaluation *****
  Num examples = 518
  Batch size = 1

***** Running Evaluation *****
  Num examples = 518
  Batch size = 1

***** Running Evaluation *****
  Num examples = 518
  Batch size = 1

***** Running Evaluation *****
  Num examples = 518
  Batch size = 1

***** Running Evaluation *****
  Num examples = 518
  Batch size = 1

***** Running Evaluation *****
  Num examples = 518
  Batch size = 1

***** Running Evaluation *****
  Num examples = 518
  Batch size = 1

***** Running Evaluation *****
  Num examples = 518
  Batch size = 1

***** Running Evaluation *****
  Num examples = 518
  Batch size = 1

***** Running Evalu

TrainOutput(global_step=616, training_loss=1.4590712367714225, metrics={'train_runtime': 4117.4646, 'train_samples_per_second': 2.391, 'train_steps_per_second': 0.15, 'total_flos': 2.436932056791552e+16, 'train_loss': 1.4590712367714225, 'epoch': 1.0})

In [ ]:
trainer.save_model("./Qwen2.5_1.5B_LoRA/final")
tokenizer.save_pretrained("./Qwen2.5_1.5B_LoRA/final")

Saving model checkpoint to ./Qwen2.5_1.5B_LoRA/final
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--unsloth--Qwen2.5-1.5B/snapshots/1582479a65dd3252951448feee6868d2cfda6452/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": null,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full

('./Qwen2.5_1.5B_LoRA/final/tokenizer_config.json',
 './Qwen2.5_1.5B_LoRA/final/tokenizer.json')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cp -r ./Qwen2.5_1.5B_LoRA /content/drive/MyDrive/